# LAB01 Data Inspection and Cleaning

This notebook works directly with the existing Python environment in VS Code. It investigates the two CSV files in `LAB01/Datasets`, documents what is inside them, and applies the cleaning steps needed for the lab tasks.

The notebook is organized around the five lab requirements:
1. Profile both files
2. Treat missing values with column-specific decisions
3. Normalize text fields and remove duplicates
4. Study the AQI distribution
5. Detect and handle extreme AQI values

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')

print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])

In [ ]:
DATA_DIR = Path(r'e:\College\MCA\Trimester 4\ML Labs\LabProgs\LAB01\Datasets')
CROP_PATH = DATA_DIR / 'crop_production.csv'
CITY_PATH = DATA_DIR / 'city_day.csv'
OUTPUT_DIR = Path(r'e:\College\MCA\Trimester 4\ML Labs\LabProgs\LAB01') / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Data directory:', DATA_DIR.resolve())
print('Crop file exists:', CROP_PATH.exists())
print('City file exists:', CITY_PATH.exists())
print('Output directory:', OUTPUT_DIR.resolve())

In [ ]:
crop_df = pd.read_csv(CROP_PATH)
city_df = pd.read_csv(CITY_PATH)

print('Crop shape:', crop_df.shape)
print('City shape:', city_df.shape)
print('\nCrop columns:\n', crop_df.columns.tolist())
print('\nCity columns:\n', city_df.columns.tolist())

print('\nCrop sample:')
display(crop_df.head())
print('\nCity sample:')
display(city_df.head())

## Task 1. Structured data profile

A data scientist should know the shape, columns, data types, null counts, duplicate counts, and whether the file contains suspicious categorical values before trusting it for modeling.

In [ ]:
def profile_frame(df: pd.DataFrame, name: str) -> dict:
    text_cols = df.select_dtypes(include='object').columns.tolist()
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    summary = {
        'dataset': name,
        'shape': df.shape,
        'columns': list(df.columns),
        'dtypes': df.dtypes.astype(str).to_dict(),
        'missing_counts': df.isna().sum().to_dict(),
        'duplicate_rows': int(df.duplicated().sum()),
        'numeric_columns': numeric_cols,
        'categorical_columns': text_cols,
    }
    return summary

crop_profile = profile_frame(crop_df, 'crop_production')
city_profile = profile_frame(city_df, 'city_day')

for profile in (crop_profile, city_profile):
    print(f"\n=== {profile['dataset']} ===")
    print('Shape:', profile['shape'])
    print('Duplicate rows:', profile['duplicate_rows'])
    print('Numeric columns:', profile['numeric_columns'])
    print('Categorical columns:', profile['categorical_columns'])
    print('Missing counts:')
    display(pd.Series(profile['missing_counts']).sort_values(ascending=False).to_frame('missing_values'))
    print('Dtypes:')
    display(pd.Series(profile['dtypes']).to_frame('dtype'))

print('Initial concern 1: city_day has several columns with substantial nulls, so blanket row deletion would waste too much data.')
print('Initial concern 2: crop_production has text fields with trailing spaces, which can break grouping and joins if not normalized.')

## Task 2. Missing-value treatment strategy

The missing-value policy below is column specific:
- numeric measurement columns are imputed with the median because it is safer than the mean for skewed pollution and production data
- target-like columns are dropped row-wise when the missingness is limited and synthetic values would be misleading
- redundant or secondary columns with too many holes are removed instead of forcing weak imputation

In [ ]:
def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'dtype': df.dtypes.astype(str),
    })
    return report[report['missing_count'] > 0].sort_values('missing_count', ascending=False)


def treat_missing_values(df: pd.DataFrame, dataset_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    cleaned = df.copy()
    report = missing_report(cleaned)
    decisions = []

    for column, row in report.iterrows():
        pct = float(row['missing_pct'])
        dtype = row['dtype']
        if column == 'AQI':
            affected = int(cleaned[column].isna().sum())
            cleaned = cleaned.dropna(subset=['AQI'])
            decisions.append((column, affected, 'drop affected rows', 'AQI is the key analysis variable; imputing it would distort the distribution and outlier checks.'))
        elif column == 'AQI_Bucket':
            affected = int(cleaned[column].isna().sum())
            cleaned = cleaned.drop(columns=['AQI_Bucket'])
            decisions.append((column, affected, 'drop column', 'AQI_Bucket is a derived label and can be reconstructed later from AQI if needed.'))
        elif column == 'Production' and dataset_name == 'crop_production':
            affected = int(cleaned[column].isna().sum())
            cleaned = cleaned.dropna(subset=[column])
            decisions.append((column, affected, 'drop affected rows', 'Production is the outcome variable for the crop table; a small number of missing rows is better removed than guessed.'))
        elif dtype == 'object':
            mode_value = cleaned[column].mode(dropna=True)
            fill_value = mode_value.iloc[0] if not mode_value.empty else 'Unknown'
            cleaned[column] = cleaned[column].fillna(fill_value)
            decisions.append((column, int(df[column].isna().sum()), 'impute with mode', f'{column} is categorical; the most frequent value is the least disruptive fill.'))
        else:
            if pct >= 25:
                cleaned = cleaned.drop(columns=[column])
                decisions.append((column, int(df[column].isna().sum()), 'drop column', f'{column} has {pct}% missingness, which is too sparse for safe imputation.'))
            else:
                if 'City' in cleaned.columns and cleaned['City'].notna().any():
                    cleaned[column] = cleaned.groupby('City')[column].transform(lambda s: s.fillna(s.median()))
                if cleaned[column].isna().any():
                    cleaned[column] = cleaned[column].fillna(cleaned[column].median())
                decisions.append((column, int(df[column].isna().sum()), 'impute with median', f'{column} is numeric; the median is robust to skew and extreme values.'))

    decision_table = pd.DataFrame(decisions, columns=['column', 'missing_before', 'action', 'reason'])
    return cleaned, decision_table

crop_clean, crop_missing_decisions = treat_missing_values(crop_df, 'crop_production')
city_clean, city_missing_decisions = treat_missing_values(city_df, 'city_day')

print('Crop missing-value decisions:')
display(crop_missing_decisions)
print('City missing-value decisions:')
display(city_missing_decisions)

print('Null counts before cleaning - crop_production:')
display(missing_report(crop_df))
print('Null counts after cleaning - crop_production:')
display(missing_report(crop_clean))
print('Null counts before cleaning - city_day:')
display(missing_report(city_df))
print('Null counts after cleaning - city_day:')
display(missing_report(city_clean))

## Task 3. State-name normalization and duplicate removal

The lab prompt refers to a `State` key, but in these files the state-like field is `State_Name` in the crop dataset. The code below normalizes all text fields, lists the raw variants that collapse to the same cleaned value, and removes duplicate records so the file is merge-ready.

In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return value
    return ' '.join(str(value).split())


def normalize_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for column in cleaned.select_dtypes(include='object').columns:
        cleaned[column] = cleaned[column].map(normalize_text)
    return cleaned


def text_variants(df: pd.DataFrame, column: str) -> pd.DataFrame:
    raw = df[column].dropna().astype(str)
    normalized = raw.map(normalize_text)
    variants = (
        pd.DataFrame({'raw': raw, 'normalized': normalized})
        .groupby('normalized')['raw']
        .agg(lambda s: sorted(set(s)))
        .reset_index()
    )
    variants['variant_count'] = variants['raw'].map(len)
    variants = variants[variants['variant_count'] > 1].sort_values('variant_count', ascending=False)
    return variants

crop_text_variants = text_variants(crop_clean, 'State_Name') if 'State_Name' in crop_clean.columns else pd.DataFrame()
print('State-name variants found in crop_production:')
display(crop_text_variants if not crop_text_variants.empty else pd.DataFrame({'message': ['No multi-spelling state-name variants found after missing-value treatment.']}))

crop_merge_ready = normalize_text_columns(crop_clean)
city_merge_ready = normalize_text_columns(city_clean)

crop_before = len(crop_merge_ready)
city_before = len(city_merge_ready)

crop_merge_ready = crop_merge_ready.drop_duplicates()
city_merge_ready = city_merge_ready.drop_duplicates()

crop_key_cols = ['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop']
if all(col in crop_merge_ready.columns for col in crop_key_cols):
    crop_merge_ready = crop_merge_ready.drop_duplicates(subset=crop_key_cols)

if all(col in city_merge_ready.columns for col in ['City', 'Date']):
    city_merge_ready = city_merge_ready.drop_duplicates(subset=['City', 'Date'])

print(f'Crop rows before: {crop_before}, after duplicate removal: {len(crop_merge_ready)}')
print(f'City rows before: {city_before}, after duplicate removal: {len(city_merge_ready)}')

print('Any remaining duplicate rows in crop?', crop_merge_ready.duplicated().sum())
print('Any remaining duplicate rows in city?', city_merge_ready.duplicated().sum())

## Task 4. AQI distribution shape

A histogram with a density curve shows where AQI values cluster, while a boxplot makes the spread and extreme values easy to see. Together they answer whether most cities sit in the middle of the AQI scale and whether a few very high readings are pulling the average upward.

In [ ]:
aqi = pd.to_numeric(city_merge_ready['AQI'], errors='coerce').dropna()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(aqi, bins=40, kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('AQI Distribution in city_day')
axes[0].set_xlabel('AQI')
axes[0].set_ylabel('Number of records')

sns.boxplot(x=aqi, ax=axes[1], color='#ff7f0e')
axes[1].set_title('AQI Boxplot')
axes[1].set_xlabel('AQI')

plt.tight_layout()
plt.show()

print('AQI summary:')
display(aqi.describe().to_frame('AQI'))
print('Skewness:', round(aqi.skew(), 3))
print('Observation 1: The histogram is right-skewed, so the average AQI is pulled upward by a long high-value tail.')
print('Observation 2: The boxplot shows several extreme readings, so the median is a safer measure of typical city AQI than the mean.')

## Task 5. Extreme AQI values

For AQI, values above 500 are implausible on the public scale, so capping is better than deleting rows. Capping preserves the record count while preventing a few extreme readings from dominating the summary statistics and the model.

In [ ]:
aqi_before = pd.to_numeric(city_merge_ready['AQI'], errors='coerce').dropna()
extreme_mask = aqi_before > 500
extreme_count = int(extreme_mask.sum())

print('Method used to detect extremes: domain cap check at AQI > 500, which is above the valid public AQI range.')
print('Extreme values found:', extreme_count)
print('Maximum AQI before treatment:', float(aqi_before.max()))
print('Mean AQI before treatment:', round(float(aqi_before.mean()), 2))

city_outlier_clean = city_merge_ready.copy()
city_outlier_clean['AQI'] = pd.to_numeric(city_outlier_clean['AQI'], errors='coerce').clip(upper=500)

print('Maximum AQI after treatment:', float(city_outlier_clean['AQI'].max()))
print('Mean AQI after treatment:', round(float(city_outlier_clean['AQI'].mean()), 2))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(aqi_before, bins=40, kde=True, ax=axes[0, 0], color='#d62728')
axes[0, 0].set_title('AQI Before Treatment')
axes[0, 0].set_xlabel('AQI')
axes[0, 0].set_ylabel('Count')

sns.histplot(city_outlier_clean['AQI'], bins=40, kde=True, ax=axes[0, 1], color='#2ca02c')
axes[0, 1].set_title('AQI After Treatment')
axes[0, 1].set_xlabel('AQI')
axes[0, 1].set_ylabel('Count')

sns.boxplot(x=aqi_before, ax=axes[1, 0], color='#d62728')
axes[1, 0].set_title('Boxplot Before Treatment')
axes[1, 0].set_xlabel('AQI')

sns.boxplot(x=city_outlier_clean['AQI'], ax=axes[1, 1], color='#2ca02c')
axes[1, 1].set_title('Boxplot After Treatment')
axes[1, 1].set_xlabel('AQI')

plt.tight_layout()
plt.show()

print('Treatment applied: AQI values above 500 were capped at 500 instead of deleting the rows.')
print('Why this works: it preserves the number of observations while removing impossible magnitudes that would distort the mean and spread.')

## Final outputs

After these steps, `crop_merge_ready` and `city_outlier_clean` are the cleaned versions to use later in the lab. The notebook keeps the original data untouched and documents each decision so the preprocessing is reproducible.

In [ ]:
print('Final crop shape:', crop_merge_ready.shape)
print('Final city shape after missing-value and duplicate handling:', city_outlier_clean.shape)
print('Final crop columns:', crop_merge_ready.columns.tolist())
print('Final city columns:', city_outlier_clean.columns.tolist())